# 实验五 QLoRA 与 DPO
使用远端实验五独立环境内核。先阅读 protocol.json 和 README.md。训练会创建新运行目录，不覆盖已有结果；本 Notebook 不包含运行输出。已有完整结果时跳过训练单元，直接读取结果。

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path(os.environ.get('LAB05_ROOT', str(Path.home() / 'llm_course_lab05')))
os.environ['LAB05_ROOT'] = str(ROOT)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
PY = ROOT / '.venv/bin/python'
CODE = ROOT / 'src/hw5_dpo.py'
def run(*args):
    subprocess.run([str(PY), str(CODE), *args], check=True)
print(json.loads((ROOT / 'src/protocol.json').read_text()))

## 1 数据、实现核验与基线
训练/评测均为指南 15 题同题回测，不是独立测试泛化。

In [ ]:
run('--mode', 'make_data')
run('--mode', 'check')
run('--mode', 'infer')

## 2 QLoRA SFT
NF4、r=8、alpha=16、batch=1、60 步；仅答案 token 计损失。

In [ ]:
run('--mode', 'sft', '--run', 'sft', '--steps', '60')
run('--mode', 'eval', '--run', 'sft')

## 3 原 12 对与新增 5 对的 DPO
两次均从同一个 SFT 适配器开始，参考模型为关闭适配器的冻结基座。

In [ ]:
for stage, pairs in [('dpo12','12'), ('dpo17','17')]:
    run('--mode','dpo','--run',stage,'--pairs',pairs,'--steps','40','--beta','0.1')
    run('--mode','eval','--run',stage)

## 4 选做 beta 扫描
beta=0.1 复用 dpo17；其余配置和训练预算相同。

In [ ]:
for stage, beta in [('beta005','0.05'), ('beta05','0.5')]:
    run('--mode','dpo','--run',stage,'--pairs','17','--steps','40','--beta',beta)
    run('--mode','eval','--run',stage)
run('--mode','summarize')

## 5 手算与真实结果读取
九条忠实度的人工分数需本人最终核验，不自动补造。

In [ ]:
for case in ['1.2,0.8,0.9,0.6,0.1','1.5,1.0,0.2,0.6,0.1','0.8,1.2,0.9,0.6,0.1']:
    run('--dpo_check', case)
print((ROOT/'outputs/all_metrics.csv').read_text(encoding='utf-8-sig'))
from IPython.display import display, Image
display(Image(filename=str(ROOT/'outputs/training_curves.png')))
display(Image(filename=str(ROOT/'outputs/accuracy_comparison.png')))

## 6 精度切换诊断（不覆盖 FP32 主结果）
若首次复现这一诊断，才执行下一单元。已有 evaluations_nf4 结果则跳过，直接读取 precision_comparison.csv。保持训练时的 NF4 基座可以帮助定位精度切换影响，不把诊断结果暗中替换主指标。

In [ ]:
d12 = 'dpo12_gc' if (ROOT/'outputs/dpo12_gc/adapter').exists() else 'dpo12'
d17 = 'dpo17_gc' if (ROOT/'outputs/dpo17_gc/adapter').exists() else 'dpo17'
for stage in ['baseline','sft',d12,d17,'beta005','beta05']:
    run('--mode','eval','--run',stage,'--eval-quantization','nf4')
subprocess.run([str(PY),str(ROOT/'src/audit_results.py'),'--outputs',str(ROOT/'outputs')],check=True)

In [ ]:
print((ROOT/'outputs/precision_comparison.csv').read_text(encoding='utf-8-sig'))
display(Image(filename=str(ROOT/'outputs/precision_comparison.png')))
print('人工复核：faithfulness_9_review.csv；CODEX 初评分与本人最终评分分开记录。')